# Data Cleaning Pipeline

This notebook performs data cleaning on the Online Retail dataset.

The objective is to remove invalid records and prepare a clean dataset for sales performance analysis.

In [2]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [3]:
df = pd.read_excel(
    "../data/raw/Online Retail.xlsx"
)

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
df.shape

(541909, 8)

# 1. Remove Duplicate Data

Duplicate records can cause inaccurate analysis because the same transaction may be counted multiple times.

Therefore, duplicate rows will be removed before performing sales analysis.

In [5]:
df.duplicated().sum()

np.int64(5268)

In [6]:
df = df.drop_duplicates()

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.shape

(536641, 8)

## Duplicate Removal Summary

The original dataset contained 541,909 transaction records.

After removing 5,268 duplicate records, the dataset contains 536,641 unique transaction records.

Duplicate removal was performed to prevent double counting during sales and revenue analysis.

# 2. Remove Cancelled Transactions

Cancelled transactions are identified by InvoiceNo starting with "C".

These records represent cancelled orders and will be removed to ensure accurate sales performance analysis.

In [9]:
cancelled_transactions = df[
    df['InvoiceNo'].astype(str).str.startswith('C')
]

cancelled_transactions.shape

(9251, 8)

In [10]:
df = df[
    ~df['InvoiceNo'].astype(str).str.startswith('C')
]

In [11]:
df.shape

(527390, 8)

In [12]:
df['InvoiceNo'].astype(str).str.startswith('C').sum()

np.int64(0)

# 3. Remove Invalid Quantity

Negative quantity values represent returned or cancelled items.

Since this project focuses on sales performance, only positive quantity transactions will be retained.

In [13]:
df[df['Quantity'] <= 0].shape

(1336, 8)

In [14]:
df[df['Quantity'] <= 0].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,NaN,United Kingdom


In [15]:
df = df[df['Quantity'] > 0]

In [16]:
df[df['Quantity'] <= 0].shape

(0, 8)

In [17]:
df.shape

(526054, 8)

## Invalid Quantity Removal Result

After removing cancelled transactions, 1,336 records with Quantity less than or equal to zero were identified.

These records were removed because negative quantity values represent returns or invalid transactions rather than actual sales.

After filtering, the dataset contains 528,654 valid transaction records.

# 4. Remove Invalid UnitPrice

Transactions with UnitPrice less than or equal to zero do not represent valid sales.

These records will be removed to ensure accurate revenue calculation.

In [18]:
df[df['UnitPrice'] <= 0].shape

(1176, 8)

In [19]:
df[df['UnitPrice'] <= 0].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom


In [20]:
df = df[df['UnitPrice'] > 0]

In [21]:
df[df['UnitPrice'] <= 0].shape

(0, 8)

In [22]:
df.shape

(524878, 8)

## Invalid UnitPrice Removal Result

The dataset contained 1,176 transactions with UnitPrice less than or equal to zero.

These records were removed because they do not represent valid sales transactions and would affect revenue calculation.

After removing invalid prices, the dataset contains 524,878 transaction records.

In [23]:
df['Description'].isnull().sum()

np.int64(0)

In [24]:
df[df['Description'].isnull()].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


In [25]:
df = df.dropna(subset=['Description'])

In [26]:
df['Description'].isnull().sum()

np.int64(0)

In [27]:
df.shape

(524878, 8)

## Missing Description Handling Result

After previous cleaning steps, no missing values were found in the Description column.

The dataset contains complete product information and is ready for further analysis.

# 6. Create Revenue Feature

Revenue is calculated by multiplying Quantity and UnitPrice.

Revenue = Quantity × UnitPrice

This feature will be used for sales performance analysis.

In [28]:
df['Revenue'] = df['Quantity'] * df['UnitPrice']

In [29]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [30]:
df['Revenue'].describe()

count    524878.000000
mean         20.275399
std         271.693566
min           0.001000
25%           3.900000
50%           9.920000
75%          17.700000
max      168469.600000
Name: Revenue, dtype: float64

In [31]:
df['Revenue'].sum()

np.float64(10642110.804000001)

# Export Clean Dataset

The cleaned dataset will be exported as a CSV file for further analysis and visualization.

In [32]:
df.to_csv(
    "../data/processed/online_retail_clean.csv",
    index=False
)

In [33]:
import os

os.path.exists("../data/processed/online_retail_clean.csv")

True